# Week 6: Working with External Data & APIs

This notebook demonstrates how to fetch real-time data from external APIs and integrate it with your datasets.

## 1. Import Required Libraries

In [2]:
import requests
import pandas as pd
import json
import time
from datetime import datetime

## 2. Understanding APIs

An API (Application Programming Interface) is a way for programs to communicate with each other and share data.

In [3]:
# Key API Concepts
api_concepts = {
    "Endpoint": "The URL where the API data is located",
    "Parameters": "Options that customize your request (country, year, format)",
    "JSON": "Data format used by most APIs (structured, human-readable)",
    "HTTP Status Codes": "Codes that tell you if the request was successful (200 = OK, 404 = Not Found)",
    "Rate Limit": "Maximum number of requests you can make per hour",
    "API Key": "Authentication token (some APIs require this, others don't)"
}

for concept, description in api_concepts.items():
    print(f"{concept}: {description}")

Endpoint: The URL where the API data is located
Parameters: Options that customize your request (country, year, format)
JSON: Data format used by most APIs (structured, human-readable)
HTTP Status Codes: Codes that tell you if the request was successful (200 = OK, 404 = Not Found)
Rate Limit: Maximum number of requests you can make per hour
API Key: Authentication token (some APIs require this, others don't)


## 3. Making Your First API Request

Let's start with a simple API that doesn't require authentication.

In [4]:
# Example: REST Countries API (no authentication required)
# This API provides information about countries

def fetch_country_data(country_name):
    """
    Fetch country information from REST Countries API
    
    Parameters:
    -----------
    country_name : str
        Name of the country (e.g., 'France')
    
    Returns:
    --------
    dict : Country data or None if request fails
    """
    try:
        # Make the API request
        url = f"https://restcountries.com/v3.1/name/{country_name}"
        response = requests.get(url)
        
        # Check if request was successful
        response.raise_for_status()
        
        # Parse JSON response
        data = response.json()
        
        return data[0] if data else None
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {country_name}: {e}")
        return None

# Test the function
france_data = fetch_country_data('France')
if france_data:
    print(f"Capital: {france_data.get('capital')}")
    print(f"Region: {france_data.get('region')}")
    print(f"Population: {france_data.get('population')}")

Capital: ['Paris']
Region: Europe
Population: 66351959


## 4. Working with JSON Data

Understanding JSON structure is key to working with APIs.

In [5]:
# Example JSON structure from an API response
example_json = """
{
  "name": "France",
  "capital": "Paris",
  "region": "Europe",
  "population": 67970571,
  "languages": {
    "fra": "French"
  },
  "borders": ["AND", "BEL", "DEU", "ITA", "LUX", "MCO", "ESP", "CHE"]
}
"""

print("Example JSON structure:")
print(example_json)

# Parse JSON string
data = json.loads(example_json)
print(f"\nAccessing data:")
print(f"Country: {data['name']}")
print(f"Capital: {data['capital']}")
print(f"Language: {data['languages']['fra']}")
print(f"Number of borders: {len(data['borders'])}")

Example JSON structure:

{
  "name": "France",
  "capital": "Paris",
  "region": "Europe",
  "population": 67970571,
  "languages": {
    "fra": "French"
  },
  "borders": ["AND", "BEL", "DEU", "ITA", "LUX", "MCO", "ESP", "CHE"]
}


Accessing data:
Country: France
Capital: Paris
Language: French
Number of borders: 8


## 5. Converting API Data to DataFrame

Transform JSON responses into tabular data for analysis.

In [6]:
def fetch_multiple_countries(countries_list):
    """
    Fetch data for multiple countries and return as DataFrame
    
    Parameters:
    -----------
    countries_list : list
        List of country names
    
    Returns:
    --------
    pd.DataFrame : Country data
    """
    data = []
    
    for country in countries_list:
        country_data = fetch_country_data(country)
        
        if country_data:
            # Extract relevant fields
            row = {
                'Country': country_data.get('name'),
                'Capital': country_data.get('capital'),
                'Region': country_data.get('region'),
                'Population': country_data.get('population'),
                'Area (km2)': country_data.get('area')
            }
            data.append(row)
        
        # Be respectful to the API - add delay between requests
        time.sleep(0.5)
    
    return pd.DataFrame(data)

# Fetch data for several countries
countries = ['France', 'Germany', 'Canada', 'Japan']
df = fetch_multiple_countries(countries)

print("Country Data from API:")
print(df)

Country Data from API:
                                             Country   Capital    Region  \
0  {'common': 'France', 'official': 'French Repub...   [Paris]    Europe   
1  {'common': 'Germany', 'official': 'Federal Rep...  [Berlin]    Europe   
2  {'common': 'Canada', 'official': 'Canada', 'na...  [Ottawa]  Americas   
3  {'common': 'Japan', 'official': 'Japan', 'nati...   [Tokyo]      Asia   

   Population  Area (km2)  
0    66351959    543908.0  
1    83491249    357114.0  
2    41651653   9984670.0  
3   123210000    377930.0  


## 6. Error Handling with APIs

Always handle potential errors when working with external data sources.

In [7]:
def safe_api_request(url, max_retries=3):
    """
    Safely make API requests with error handling and retries
    
    Parameters:
    -----------
    url : str
        The API endpoint URL
    max_retries : int
        Maximum number of retry attempts
    
    Returns:
    --------
    dict or None : JSON response or None if request fails
    """
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return response.json()
            
        except requests.exceptions.Timeout:
            print(f"Request timed out. Attempt {attempt + 1} of {max_retries}")
            
        except requests.exceptions.ConnectionError:
            print(f"Connection error. Is the API server down? Attempt {attempt + 1} of {max_retries}")
            
        except requests.exceptions.HTTPError as e:
            if response.status_code == 404:
                print(f"Resource not found (404)")
                return None
            elif response.status_code == 429:
                print(f"Rate limit exceeded. Waiting before retry...")
                time.sleep(2)
            else:
                print(f"HTTP Error: {e}")
                
        except json.JSONDecodeError:
            print(f"Could not parse JSON response")
            return None
            
    print(f"Failed after {max_retries} attempts")
    return None

# Test error handling
print("Testing error handling:")
print("Attempting to fetch country 'InvalidCountry'...")
result = safe_api_request("https://restcountries.com/v3.1/name/InvalidCountryXYZ")

Testing error handling:
Attempting to fetch country 'InvalidCountry'...
Resource not found (404)


## 7. Best Practices for API Integration

Follow these guidelines when working with APIs.

In [8]:
best_practices = {
    "DO": [
        "✓ Check API documentation before making requests",
        "✓ Cache responses locally to reduce API calls",
        "✓ Use try/except blocks for error handling",
        "✓ Respect rate limits - add delays between requests",
        "✓ Log requests for debugging",
        "✓ Test your code with small datasets first"
    ],
    "DON'T": [
        "✗ Hardcode API keys in your code",
        "✗ Make unnecessary API calls",
        "✗ Share API responses publicly (may contain sensitive data)",
        "✗ Ignore rate limits",
        "✗ Assume data will always be in the expected format",
        "✗ Make assumptions about data quality"
    ]
}

for category, items in best_practices.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  {item}")


DO:
  ✓ Check API documentation before making requests
  ✓ Cache responses locally to reduce API calls
  ✓ Use try/except blocks for error handling
  ✓ Respect rate limits - add delays between requests
  ✓ Log requests for debugging
  ✓ Test your code with small datasets first

DON'T:
  ✗ Hardcode API keys in your code
  ✗ Make unnecessary API calls
  ✗ Share API responses publicly (may contain sensitive data)
  ✗ Ignore rate limits
  ✗ Assume data will always be in the expected format
  ✗ Make assumptions about data quality


## 10. Assignment: Fetch and Merge API Data

**Task**: Create a notebook that fetches data from a public API and merges it with an existing dataset.

**Requirements**:
1. **Choose an API** from the recommended list (or find your own)
2. **Make API requests** to fetch relevant data
3. **Convert JSON to DataFrame** with proper column names
4. **Handle errors gracefully** with try/except blocks
5. **Merge with existing data** to create a richer dataset
6. **Analyze the merged data** and share insights
7. **Document your process** with clear explanations

**Tips**:
- Start with a simple API like REST Countries
- Test with a single country/record first
- Add error handling for missing data
- Respect API rate limits - don't hammer the server!
- Cache successful responses locally